# 4. Predictive/ Forecasting Tools

**Goal:** Use the DataRobot forecast model deployement as a tool.

**Key Concept:**
LLMs cannot predict the future, but DataRobot can. In this notebook, we write a **Tool Client** function. This wrapper:
1.  **Defines a Schema:** Forces the LLM to extract structured inputs (e.g., a specific date and item).
2.  **Activates:** Triggers only when the user asks a forward-looking question.
3.  **Predicts:** Queries a deployed Time Series model to return an accurate forecast number to the chat.

In [1]:
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Initialize Client
load_dotenv()
dr_client = dr.Client()

# 2. Configure the Tool Connection (MCP)
# We swap in your forecasting deployment ID here
MCP_DEPLOYMENT_ID = "692eee8f6362220c3956af6e" 

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

# 3. Configure the Model (The Brain)
# Using the same Azure GPT-5 configuration via the DataRobot Gateway
MODEL_NAME = "azure/gpt-5-2025-08-07"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

# 4. Define the Agent
# We update the prompt to ensure the agent understands its role is forecasting
system_prompt = """
You are a helpful forecasting assistant. 
Use the available forecasting tools to predict future values based on the user's request.
"""
agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

# 5. Run the Forecast
# The agent will detect the forecast tool exposed by the MCP server and use it
async with server:
    response = await agent.run("Please generate a forecast using the scoring dataset '6972a732b304ef35e947921a' and the deployment '6972a516e0ccfc2a8382e0e4' and find the forecast for 2025-10-27 whole day.")
    pprint(response.output)

('I generated the forecast using your scoring dataset and deployment, aligning '
 'the data to the deployment’s daily granularity. For the day 2025-10-27 '
 '(UTC), the predicted values by hub are:\n'
 '\n'
 '- HB_HOUSTON: -42957.070312\n'
 '- HB_NORTH: -42958.527344\n'
 '- HB_SOUTH: -42955.855469\n'
 '- HB_WEST: -42957.339844\n'
 '\n'
 'Notes:\n'
 '- These are daily forecasts (one value per hub for the whole day).\n'
 '- The predictions dataset reference is 49c2e, with timestamp_utc = '
 '2025-10-27.\n'
 '- If you’d like an aggregate across hubs or clarification on the target’s '
 'units/sign, I can provide that.')


In [11]:
async with server:
    response = await agent.run("Please list the forecasting models or deployments you currently have access to. For each one, provide the Deployment ID and the target name if available")
    pprint(response.output)

('Here’s what I can access right now.\n'
 '\n'
 'Deployed forecasting models:\n'
 '- Time Series (Jan-8-2026)\n'
 '  - Deployment name: UNITS (actual) predictions 2026-01-22 05:16\n'
 '  - Deployment ID: 6971b39b3fa6dde87d114a82\n'
 '  - Target: UNITS (actual)\n'
 '\n'
 'Other deployments (non-forecasting):\n'
 '- Generative Analyst Deployment [test_TTMDD1]\n'
 '  - Deployment ID: 680e8533970d7346f881fbbc\n'
 '  - Target: not applicable (generative, not forecasting)\n'
 '\n'
 'Forecasting models (projects not currently deployed):\n'
 '- Business Planner Default Use Case\n'
 '  - Project ID: 6969272e24fa8a24f828ab98\n'
 '  - Target: dam_price_usd_mwh (actual)\n'
 '- Time Series (Jan-8-2026)\n'
 '  - Project ID: 69602db64a2f4593ad6a0956\n'
 '  - Target: UNITS (actual)\n'
 '- TS\n'
 '  - Project ID: 696161e72223ddf56a716201\n'
 '  - Target: GrossProfit (actual)\n'
 '  - Project ID: 695ffa9ab25e79fbea716805\n'
 '  - Target: GrossProfit (actual)\n'
 '- Peddle Time Series Modeling\n'
 '  - P

In [0]:
async with server:
    response = await agent.run("Using the customer record for John Doe from dataset 69724dfb1455f54a494792b1, generate a prediction using the deployed model 6967d7d4d467b29e341daa65 and report John Doe’s probability of default.")
    pprint(response.output)

## Using DR managed prompts 

In [5]:
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from datarobot.models.genai.prompt_template import PromptTemplate
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Initialize Client
load_dotenv()
dr_client = dr.Client()

# 2. Configure the Tool Connection (MCP)
MCP_DEPLOYMENT_ID = "692eee8f6362220c3956af6e" 

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

# --- NEW STEP: PREPARE PROMPT CONTEXT ---
# We fetch the deployment details so we can inject the real name into the prompt
deployment = dr.Deployment.get(MCP_DEPLOYMENT_ID)
deployment_name = deployment.label  # e.g., "Sales Forecast Production"

# 3. Fetch and Render System Prompt
# PASTE YOUR TEMPLATE ID HERE (from DataRobot UI)
PROMPT_TEMPLATE_ID = "6972c21dd3030179e93b8761" 

# OPTIONAL: Set to specific version like "v1", or None for latest
PROMPT_VERSION_ID = "v2" 

print(f"--- Setting up Agent for Deployment: {deployment_name} ---")

# Fetch Template
template = PromptTemplate.get(PROMPT_TEMPLATE_ID)

# Determine Version (Smart Logic)
target_version = None
if PROMPT_VERSION_ID:
    versions = template.list_versions()
    search_str = str(PROMPT_VERSION_ID).lower().replace("v", "")
    for v in versions:
        if v.id == PROMPT_VERSION_ID:
            target_version = v
            break
        if hasattr(v, 'version') and str(v.version) == search_str:
            target_version = v
            break
    if not target_version:
        raise ValueError(f"Version '{PROMPT_VERSION_ID}' not found.")
else:
    target_version = template.get_latest_version()

print(f"Using Prompt Version: v{getattr(target_version, 'version', '?')} (ID: {target_version.id})")

# Render Prompt with Dynamic Variables
# This ensures the agent knows exactly which deployment it is talking about
try:
    system_prompt = target_version.render(
        variables={
            "deployment_name": deployment_name,
            # Add "company_name" here if your specific template still requires it
            # "company_name": "MyCompany" 
        }
    )
except Exception as e:
    print(f"Error rendering prompt. Ensure your UI template uses {{deployment_name}}.")
    raise e

# 4. Configure the Model (The Brain)
MODEL_NAME = "azure/gpt-5-2025-08-07"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

# 5. Define the Agent
# Now using the dynamic 'system_prompt' from DataRobot
agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

# 6. Run the Forecast
async with server:
    print("\n--- Running Forecast Request ---")
    response = await agent.run("Generate forecast for 2022-10-25")
    pprint(response.output)

--- Setting up Agent for Deployment: MCP Server [wren-mcp-LMR-Dec-2025] ---
Using Prompt Version: v2 (ID: 6972c58cd3030179e93b876c)

--- Running Forecast Request ---
('Here are the forecasts from the deployment for forecast point 2022-10-25 '
 '(series: store_182_SKU_56889087), covering the next four weekly periods:\n'
 '\n'
 '- 2022-11-01: 59.62 units\n'
 '- 2022-11-08: 58.82 units\n'
 '- 2022-11-15: 58.35 units\n'
 '- 2022-11-22: 57.13 units\n'
 '\n'
 'Note: The original scoring data did not include future rows beyond the '
 'forecast point, so I appended future timestamps through 2022-11-22 to enable '
 'the predictions. If you’d like a longer horizon or scenario adjustments '
 '(e.g., price or promo changes), I can apply those and regenerate the '
 'forecast.')
